# Time Series Anomaly Detection

## Loading data

In [1]:
# Connecting to kaggle
from google.colab import files
files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle API configured successfully")

Saving kaggle.json to kaggle.json
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Kaggle API configured successfully


!kaggle datasets download -d username/dataset-name

!unzip filename.zip -d destination_folder

In [2]:
# Loading Dataset
!kaggle datasets download -d brjapon/cwru-bearing-datasets

Dataset URL: https://www.kaggle.com/datasets/brjapon/cwru-bearing-datasets
License(s): CC-BY-SA-4.0
100% 40.4M/40.4M [00:03<00:00, 12.9MB/s]



In [3]:
# Unzipping
!unzip /content/cwru-bearing-datasets.zip -d /content/cwru-bearing
print("Dataset ready")

Archive:  /content/cwru-bearing-datasets.zip
  inflating: /content/cwru-bearing/CWRU_48k_load_1_CNN_data.npz  
  inflating: /content/cwru-bearing/feature_time_48k_2048_load_1.csv  
  inflating: /content/cwru-bearing/raw/B007_1_123.mat  
  inflating: /content/cwru-bearing/raw/B014_1_190.mat  
  inflating: /content/cwru-bearing/raw/B021_1_227.mat  
  inflating: /content/cwru-bearing/raw/IR007_1_110.mat  
  inflating: /content/cwru-bearing/raw/IR014_1_175.mat  
  inflating: /content/cwru-bearing/raw/IR021_1_214.mat  
  inflating: /content/cwru-bearing/raw/OR007_6_1_136.mat  
  inflating: /content/cwru-bearing/raw/OR014_6_1_202.mat  
  inflating: /content/cwru-bearing/raw/OR021_6_1_239.mat  
  inflating: /content/cwru-bearing/raw/Time_Normal_1_098.mat  
Dataset ready


## Exploring Data

In [4]:
os.listdir('/content/cwru-bearing')

['CWRU_48k_load_1_CNN_data.npz', 'raw', 'feature_time_48k_2048_load_1.csv']

- raw: likely a folder with raw vibration signal files. This is the most fundamental form of the data.
- CWRU_48k_load_1_CNN_data.npz: preprocessed data already formatted for CNN input. Someone has already done the feature extraction.
- feature_time_48k_2048_load_1.csv: time domain features already extracted into a

In [5]:
# Checking the raw file
os.listdir('/content/cwru-bearing/raw')

['OR014_6_1_202.mat',
 'B007_1_123.mat',
 'OR021_6_1_239.mat',
 'B021_1_227.mat',
 'IR014_1_175.mat',
 'Time_Normal_1_098.mat',
 'IR007_1_110.mat',
 'IR021_1_214.mat',
 'OR007_6_1_136.mat',
 'B014_1_190.mat']

The raw folder contains .mat files - MATLAB format.
- Time_Normal_1_098 - normal bearing data
- IR007, IR014, IR021 - Inner Race faults at different severities
- OR007, OR014, OR021 - Outer Race faults
- B007, B014, B021 - Ball faults

In [6]:
# Checking csv data file info
import pandas as pd
data = pd.read_csv('/content/cwru-bearing/feature_time_48k_2048_load_1.csv')

data

,max,min,mean,sd,rms,skewness,kurtosis,crest,form,fault
0,0.35986,-0.41890,0.017840,0.122746,0.124006,-0.118571,-0.042219,2.901946,6.950855,Ball_007_1
1,0.46772,-0.36111,0.022255,0.132488,0.134312,0.174699,-0.081548,3.482334,6.035202,Ball_007_1
2,0.46855,-0.43809,0.020470,0.149651,0.151008,0.040339,-0.274069,3.102819,7.376926,Ball_007_1
3,0.58475,-0.54303,0.020960,0.157067,0.158422,-0.023266,0.134692,3.691097,7.558387,Ball_007_1
4,0.44685,-0.57891,0.022167,0.138189,0.139922,-0.081534,0.402783,3.193561,6.312085,Ball_007_1
...,...,...,...,...,...,...,...,...,...,...
2295,0.21425,-0.19839,0.010769,0.064100,0.064983,-0.212497,-0.119312,3.297037,6.034174,Normal_1
2296,0.21967,-0.20882,0.013136,0.068654,0.069883,-0.061308,-0.295122,3.143410,5.319958,Normal_1
2297,0.20799,-0.21613,0.012571,0.067128,0.068279,-0.154754,-0.071405,3.046161,5.431299,Normal_1
2298,0.21425,-0.22405,0.012608,0.066813,0.067977,-0.326966,0.023662,3.151821,5.391672,Normal_1


2,300 rows, 10 columns, 9 feature columns and 1 fault label column.

## Checking CSV data Feature
I decided to use the csv file as it is already prepared and my focus for this project is Anomaly detection and not signal processing.

In [7]:
data['fault'].unique()    #Checking unique fault column features

array(['Ball_007_1', 'Ball_014_1', 'Ball_021_1', 'IR_007_1', 'IR_014_1',
       'IR_021_1', 'OR_007_6_1', 'OR_014_6_1', 'OR_021_6_1', 'Normal_1'],
      dtype=object)

In [8]:
data['fault'].nunique()

10

In [9]:
data['fault'].value_counts()

,count
fault,
Ball_007_1,230
Ball_014_1,230
Ball_021_1,230
IR_007_1,230
IR_014_1,230
IR_021_1,230
OR_007_6_1,230
OR_014_6_1,230
OR_021_6_1,230


## Data preparation and cleaning

In [10]:
Normal = data[data['fault'] == 'Normal_1']
Faulty = data[data['fault'] != 'Normal_1']

print(Normal.shape)
print(Faulty.shape)

print(Normal)
print(Faulty)

(230, 10)
(2070, 10)
          max      min      mean        sd       rms  skewness  kurtosis  \
2070  0.20423 -0.19881  0.008002  0.066461  0.066925 -0.277750 -0.182979   
2071  0.20444 -0.19735  0.009596  0.063003  0.063714 -0.272165 -0.019003   
2072  0.21133 -0.18024  0.010816  0.063067  0.063973 -0.210367 -0.196338   
2073  0.20090 -0.19672  0.011418  0.064289  0.065280 -0.096509  0.103004   
2074  0.21154 -0.22781  0.012309  0.063622  0.064787 -0.225488 -0.034797   
...       ...      ...       ...       ...       ...       ...       ...   
2295  0.21425 -0.19839  0.010769  0.064100  0.064983 -0.212497 -0.119312   
2296  0.21967 -0.20882  0.013136  0.068654  0.069883 -0.061308 -0.295122   
2297  0.20799 -0.21613  0.012571  0.067128  0.068279 -0.154754 -0.071405   
2298  0.21425 -0.22405  0.012608  0.066813  0.067977 -0.326966  0.023662   
2299  0.19610 -0.24721  0.012209  0.063243  0.064396 -0.351762  0.226294   

         crest      form     fault  
2070  3.051634  8.363554  Nor

In [11]:
Normal2 = Normal.drop(columns=['fault'])
Faulty2 = Faulty.drop(columns=['fault'])

print(Normal2.shape)
print(Faulty2.shape)

print(Normal2)
print(Faulty2)

(230, 9)
(2070, 9)
          max      min      mean        sd       rms  skewness  kurtosis  \
2070  0.20423 -0.19881  0.008002  0.066461  0.066925 -0.277750 -0.182979   
2071  0.20444 -0.19735  0.009596  0.063003  0.063714 -0.272165 -0.019003   
2072  0.21133 -0.18024  0.010816  0.063067  0.063973 -0.210367 -0.196338   
2073  0.20090 -0.19672  0.011418  0.064289  0.065280 -0.096509  0.103004   
2074  0.21154 -0.22781  0.012309  0.063622  0.064787 -0.225488 -0.034797   
...       ...      ...       ...       ...       ...       ...       ...   
2295  0.21425 -0.19839  0.010769  0.064100  0.064983 -0.212497 -0.119312   
2296  0.21967 -0.20882  0.013136  0.068654  0.069883 -0.061308 -0.295122   
2297  0.20799 -0.21613  0.012571  0.067128  0.068279 -0.154754 -0.071405   
2298  0.21425 -0.22405  0.012608  0.066813  0.067977 -0.326966  0.023662   
2299  0.19610 -0.24721  0.012209  0.063243  0.064396 -0.351762  0.226294   

         crest      form  
2070  3.051634  8.363554  
2071  3.208700

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
scaled_normal = scaler.fit_transform(Normal2)
scaled_faulty = scaler.transform(Faulty2)

x_train, x_val = train_test_split(scaled_normal, test_size=0.2, random_state=42)

print(scaled_normal)
print(scaled_faulty)

[[-0.04550852  0.35002136 -1.77845302 ... -0.55378541 -0.16811885
   2.43245719]
 [-0.03443795  0.41653853 -1.14655182 ...  0.48577519  0.45521687
   0.95308642]
 [ 0.32878212  1.19606512 -0.66289888 ... -0.63847789  0.83124384
   0.33083891]
 ...
 [ 0.15270737 -0.43907276  0.03320539 ...  0.15356    -0.1898382
  -0.08410391]
 [ 0.48271573 -0.79990564  0.04757152 ...  0.75625989  0.22948424
  -0.11811348]
 [-0.47409765 -1.85506845 -0.11046589 ...  2.04088639 -0.19347906
  -0.21876684]]
[[ 8.15883677e+00 -9.67721450e+00  2.12218798e+00 ...  3.38590757e-01
  -7.62172613e-01  1.22003091e+00]
 [ 1.38448914e+01 -7.04431905e+00  3.87232650e+00 ...  8.92608869e-02
   1.54116232e+00  4.34186645e-01]
 [ 1.38886465e+01 -1.05515053e+01  3.16483013e+00 ... -1.13126692e+00
   3.50140169e-02  1.58569971e+00]
 ...
 [ 2.55261297e+02 -2.31493759e+02  2.88275667e-01 ...  1.49603696e+02
   1.85434569e+01  3.74599588e+01]
 [ 2.12054451e+02 -1.71373170e+02  3.58421453e-01 ...  6.94565112e+01
   1.43764686e

There is no y in an autoencoder.There is no y in an autoencoder.

In classification, y is the label, what the model is trying to predict.

In [16]:
print(x_train.shape)
print(x_val.shape)

(184, 9)
(46, 9)


## Training Model

In [17]:
# Importing Libaries
import torch.nn as nn

In [22]:
# Autoencoder Layer
import torch.nn as nn

class Autoencoder(nn.Module):
  def __init__(self):
    super(Autoencoder, self).__init__()

    # Encoder
    self.encoder = nn.Sequential(
        nn.Linear(9, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 4),
    )

    # Decoder
    self.decoder = nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 16),
        nn.ReLU(),
        nn.Linear(16, 9),
    )

  def forward(self, x):
    x = self.encoder(x)
    x = self.decoder(x)
    return x

In [23]:
model = Autoencoder()
print(model)

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=9, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=4, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=9, bias=True)
  )
)


In [25]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [26]:
# Features
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
'''
# Training
num_epochs = 20

x_train = torch.FloatTensor(x_train)
for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0

  for inputs in x_train:
    optimizer.zero_grad()
    outputs = model()
    loss = criterion(outputs, inputs)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}')

In [ ]:
'''
# Validation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

In [ ]:
'''
# Plotting Accuracies
import matplotlib.pyplot as plt

train_losses = [1.9143, 1.6107, 1.4812, 1.3868, 1.3217, 1.2675,
                1.2137, 1.1670, 1.1273, 1.0877, 1.0533, 1.0242,
                0.9909, 0.9666, 0.9382, 0.9101, 0.8920, 0.8615,
                0.8415, 0.8196]

plt.figure(figsize=(10, 5))
plt.plot(range(1, 21), train_losses, marker='o', color='steelblue')
plt.title('Training Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.savefig('training_loss.png', dpi=150)
plt.show()